# Additional Checks: (A) Leakage Bias for All Six Models, (B) Hyperparameter-Boundary Sensitivity

**Run after the main notebook.** Two reviewer-style questions are answered:

| Section | Question |
|---|---|
| A | The main notebook measured the optimistic bias of subject-mixing CV only for the linear SVM. Is the bias larger for more flexible models (KNN, Decision Tree, Random Forest, Gradient Boosting)? |
| B | Several selected hyperparameters lie at the edge of the search grid (KNN k=21, RBF C=100, RF min_samples_leaf=3, GB learning_rate=0.1). Does extending the grid change the conclusions? |

All tuning in Section B uses the **training subjects only** (subject-grouped 5-fold CV); the test subjects are used only to report the
effect of the extended search on the held-out result. Outputs are numbered `Ext14...` / `Ext15...` and downloaded as one ZIP.

## 0. Setup

In [ ]:
!pip install -q kagglehub

In [ ]:
import os, time, json, shutil, warnings, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import sklearn, scipy
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")

# ------------------------- USER SETTINGS -------------------------
SEED = 42
FAST_MODE = False            # True: quick trial run
N_REPEATS = 1 if FAST_MODE else 3   # repetitions of the naive vs. grouped comparison (different seeds)
RF_TREES = 50 if FAST_MODE else 300
HGB_ITER = 30 if FAST_MODE else 200
RUN_LEAKAGE_ALL_MODELS = True
RUN_GRID_SENSITIVITY = True
# ----------------------------------------------------------------
OUT_DIR = Path("har_checks_outputs"); (OUT_DIR / "tables").mkdir(parents=True, exist_ok=True); (OUT_DIR / "figures").mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook"); plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight"})
T0 = time.time()

def save_table(df, name, caption=""):
    df.to_csv(OUT_DIR / "tables" / f"{name}.csv")
    try: df.to_latex(OUT_DIR / "tables" / f"{name}.tex", float_format="%.4f", caption=caption or None, label=f"tab:{name}")
    except Exception: pass

def savefig(name):
    plt.savefig(OUT_DIR / "figures" / f"{name}.png"); plt.savefig(OUT_DIR / "figures" / f"{name}.pdf"); plt.show()

## 1. Data Loading

In [ ]:
import kagglehub
path = kagglehub.dataset_download("uciml/human-activity-recognition-with-smartphones")
csv_files = {p.name.lower(): p for p in Path(path).rglob("*.csv")}
train_df = pd.read_csv(csv_files["train.csv"]); test_df = pd.read_csv(csv_files["test.csv"])
print("Train:", train_df.shape, "| Test:", test_df.shape)

In [ ]:
TARGET, GROUP = "Activity", "subject"
feature_cols = [c for c in train_df.columns if c not in (TARGET, GROUP)]
assert not (set(train_df[GROUP]) & set(test_df[GROUP]))
le = LabelEncoder().fit(train_df[TARGET])
X_train = train_df[feature_cols].to_numpy(float); y_train = le.transform(train_df[TARGET]); g_train = train_df[GROUP].to_numpy()
X_test = test_df[feature_cols].to_numpy(float); y_test = le.transform(test_df[TARGET])

def pipe(clf): return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

# Best hyperparameters of the main run and of the extension run (Ext01)
MODELS = {
    "SVM (Linear)":      pipe(SVC(kernel="linear", C=0.1, random_state=SEED)),
    "SVM (RBF)":         pipe(SVC(kernel="rbf", C=100, gamma=0.001, random_state=SEED)),
    "KNN":               pipe(KNeighborsClassifier(n_neighbors=21, weights="distance", metric="manhattan")),
    "Decision Tree":     pipe(DecisionTreeClassifier(criterion="entropy", max_depth=12, min_samples_leaf=5, random_state=SEED)),
    "Random Forest":     pipe(RandomForestClassifier(n_estimators=RF_TREES, max_features="sqrt", min_samples_leaf=3, n_jobs=-1, random_state=SEED)),
    "Gradient Boosting": pipe(HistGradientBoostingClassifier(max_iter=HGB_ITER, learning_rate=0.1, l2_regularization=0.0, random_state=SEED)),
}
NAMES = list(MODELS)

## A. Optimistic Bias of Subject-Mixing CV for All Six Models

Same design as Table 2b of the main notebook, but for every model (fixed hyperparameters, **training subjects only**):
naive random stratified 5-fold (subject-mixing, leaky) versus subject-grouped 5-fold. The bias is the difference of the mean macro-F1.

In [ ]:
def leak_fold(name, scheme, tr, te, rep, fold):
    m = clone(MODELS[name]).fit(X_train[tr], y_train[tr]); yp = m.predict(X_train[te])
    return {"Model": name, "Scheme": scheme, "Repeat": rep, "Fold": fold,
            "Macro-F1": f1_score(y_train[te], yp, average="macro", zero_division=0), "Accuracy": accuracy_score(y_train[te], yp)}

if RUN_LEAKAGE_ALL_MODELS:
    t = time.time(); jobs = []
    for rep in range(N_REPEATS):
        naive = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + rep)
        grouped = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + rep)
        for scheme, splitter, kw in [("Naive", naive, {}), ("Grouped", grouped, {"groups": g_train})]:
            for fold, (tr, te) in enumerate(splitter.split(X_train, y_train, **kw)):
                for name in NAMES: jobs.append((name, scheme, tr, te, rep, fold))
    leak = pd.DataFrame(Parallel(n_jobs=-1)(delayed(leak_fold)(*j) for j in jobs)); print(f"{len(jobs)} fits in {time.time()-t:.0f}s")
    save_table(leak.set_index(["Model", "Scheme", "Repeat", "Fold"]), "Ext14a_leakage_all_models_fold_scores")
    g = leak.groupby(["Model", "Scheme"])["Macro-F1"].agg(["mean", "std"]).unstack("Scheme")
    out = pd.DataFrame({"Naive macro-F1 (mean)": g[("mean", "Naive")], "Naive (std over folds)": g[("std", "Naive")],
                        "Grouped macro-F1 (mean)": g[("mean", "Grouped")], "Grouped (std over folds)": g[("std", "Grouped")]}).loc[NAMES]
    out["Optimistic bias (points)"] = (out["Naive macro-F1 (mean)"] - out["Grouped macro-F1 (mean)"]) * 100
    per_rep = leak.groupby(["Model", "Scheme", "Repeat"])["Macro-F1"].mean().unstack("Scheme")
    out["Bias std over repeats (points)"] = ((per_rep["Naive"] - per_rep["Grouped"]) * 100).groupby("Model").std().loc[NAMES]
    display(out.round(4)); save_table(out, "Ext14_leakage_bias_all_models", caption="Optimistic bias of naive (subject-mixing) vs. subject-grouped 5-fold CV for all models.")

    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.barh(NAMES[::-1], out["Optimistic bias (points)"].loc[NAMES[::-1]], color="tab:red")
    for i, v in enumerate(out["Optimistic bias (points)"].loc[NAMES[::-1]]): ax.text(v + 0.1, i, f"{v:.1f}", va="center")
    ax.set_xlabel("Optimistic bias of naive CV (macro-F1 points)"); ax.set_title("Bias of subject-mixing CV by model")
    savefig("FigureExt05_leakage_bias_all_models")
else:
    print("Section A skipped.")

## B. Sensitivity to the Hyperparameter-Grid Boundary

For every model whose selected value was at the edge of the original grid, the search is extended **beyond** the old edge
(the original selected value is kept inside the new grid so that it is directly comparable). Search: subject-grouped 5-fold CV,
macro-F1, training subjects only. The effect on the held-out test subjects is reported afterwards (not used for selection).

In [ ]:
cv5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
EXT_GRIDS = {
    "KNN":               {"clf__n_neighbors": [21, 31, 51, 75]},
    "SVM (RBF)":         {"clf__C": [100, 300, 1000], "clf__gamma": [1e-4, 1e-3]},
    "Random Forest":     {"clf__min_samples_leaf": [3, 5, 10]},
    "Gradient Boosting": {"clf__learning_rate": [0.1, 0.2, 0.3]},
}
ORIG = {"KNN": {"clf__n_neighbors": 21}, "SVM (RBF)": {"clf__C": 100, "clf__gamma": 1e-3},
        "Random Forest": {"clf__min_samples_leaf": 3}, "Gradient Boosting": {"clf__learning_rate": 0.1}}

if RUN_GRID_SENSITIVITY:
    rows = []
    for name, grid in EXT_GRIDS.items():
        t = time.time()
        gs = GridSearchCV(MODELS[name], grid, scoring="f1_macro", cv=cv5, n_jobs=1 if name == "Random Forest" else -1, refit=False)
        gs.fit(X_train, y_train, groups=g_train)
        res = pd.DataFrame(gs.cv_results_); best_i = int(gs.best_index_)
        orig_i = [i for i, p in enumerate(res["params"]) if all(p[k] == v for k, v in ORIG[name].items())][0]
        best_p = res["params"][best_i]
        # edge check: is the new optimum again at the largest value of the extended grid?
        at_edge = all(best_p[k] == max(v) for k, v in grid.items() if len(v) > 1 and k in ("clf__n_neighbors", "clf__C", "clf__min_samples_leaf", "clf__learning_rate"))
        accs = {}
        for tag, params in [("orig", res["params"][orig_i]), ("best", best_p)]:
            m = clone(MODELS[name]).set_params(**params).fit(X_train, y_train); yp = m.predict(X_test)
            accs[tag] = (accuracy_score(y_test, yp), f1_score(y_test, yp, average="macro"))
        rows.append({"Model": name, "Original": ", ".join(f"{k.replace('clf__','')}={v}" for k, v in ORIG[name].items()),
                     "Best in extended grid": ", ".join(f"{k.replace('clf__','')}={v}" for k, v in best_p.items()),
                     "CV macro-F1 original": res["mean_test_score"][orig_i], "CV macro-F1 best": res["mean_test_score"][best_i],
                     "CV gain (points)": (res["mean_test_score"][best_i] - res["mean_test_score"][orig_i]) * 100,
                     "Test acc original": accs["orig"][0], "Test acc best": accs["best"][0], "Test acc change (points)": (accs["best"][0] - accs["orig"][0]) * 100,
                     "Optimum still at new edge": bool(at_edge), "Search time (s)": round(time.time() - t, 1)})
        print(name, "->", best_p, f"({time.time()-t:.0f}s)")
        res[["params", "mean_test_score", "std_test_score"]].to_csv(OUT_DIR / "tables" / f"Ext15_grid_{name.replace(' ', '_').replace('(', '').replace(')', '')}.csv", index=False)
    sens = pd.DataFrame(rows).set_index("Model"); display(sens.round(4))
    save_table(sens, "Ext15_extended_grid_sensitivity", caption="Effect of extending the hyperparameter grids beyond the original edge.")
else:
    print("Section B skipped.")

## C. Save and Download

In [ ]:
cfg = {"env": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
               "scipy": scipy.__version__, "seed": SEED},
       "settings": {"FAST_MODE": FAST_MODE, "N_REPEATS": N_REPEATS, "RF_TREES": RF_TREES, "HGB_ITER": HGB_ITER},
       "runtime_seconds": round(time.time() - T0, 1)}
(OUT_DIR / "config_and_environment_checks.json").write_text(json.dumps(cfg, indent=2))
zip_path = shutil.make_archive("har_checks_outputs", "zip", OUT_DIR)
print("ZIP:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB), runtime {time.time()-T0:.0f}s")
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Not running in Colab; download the ZIP manually.")